# 🦟🧬 Integração InfoDengue + Pathoplexus: Dados Históricos do Brasil (1986-2012)

Este notebook demonstra a análise integrada de dados epidemiológicos e genômicos da dengue no Brasil, utilizando dados históricos disponíveis.

## 📊 Dados Disponíveis

### Pathoplexus - Sequências Genômicas
- **Período:** 1986-2012
- **Total de sequências:** 7,643 (Brasil)
- **Cobertura:** Genomas com metadados de sorotipo, local e data
- **API:** https://lapis.pathoplexus.org/dengue

### InfoDengue (via Mosqlimate)
- **Dados:** Notificações de dengue (SINAN)
- **Cobertura:** Casos suspeitos e confirmados por município/semana
- **API:** https://api.mosqlimate.org/

## 🎯 Objetivos da Análise
- Analisar a distribuição temporal dos sorotipos ao longo de 26 anos
- Identificar padrões de circulação de DENV-1, DENV-2, DENV-3 e DENV-4
- Visualizar a cobertura geográfica das amostras sequenciadas
- Demonstrar a integração entre dados epidemiológicos e genômicos

In [ ]:
# Setup e imports
import sys
sys.path.insert(0, '../../scripts/accessors')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import requests
import warnings
warnings.filterwarnings('ignore')

from pathoplexus import PathoplexusAccessor

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✅ Bibliotecas carregadas com sucesso!")
print(f"Data atual: {datetime.now().strftime('%Y-%m-%d')}")

## 1. Carregar Dados Genômicos do Pathoplexus

In [ ]:
print("🧬 Carregando dados genômicos do Pathoplexus...")
print("="*60)

# Inicializar Pathoplexus
ppx = PathoplexusAccessor('dengue')

# Buscar todos os metadados do Brasil
metadata = ppx.get_metadata(
    country='Brazil',
    limit=10000,  # Limite alto para pegar todos
)

print(f"✅ {len(metadata)} sequências carregadas")

# Processar datas
if not metadata.empty and 'sampleCollectionDate' in metadata.columns:
    metadata['sampleCollectionDate'] = pd.to_datetime(metadata['sampleCollectionDate'], errors='coerce')
    metadata['year'] = metadata['sampleCollectionDate'].dt.year
    metadata['month'] = metadata['sampleCollectionDate'].dt.month
    metadata['decade'] = (metadata['year'] // 10) * 10
    
    print(f"\n📅 Range temporal:")
    print(f"  De: {metadata['sampleCollectionDate'].min().strftime('%Y-%m-%d')}")
    print(f"  Até: {metadata['sampleCollectionDate'].max().strftime('%Y-%m-%d')}")
    print(f"  Total de anos: {metadata['year'].max() - metadata['year'].min() + 1}")

print("\n📊 Primeiras linhas:")
display_cols = ['accession', 'sampleCollectionDate', 'geoLocAdmin1', 'serotype']
print(metadata[display_cols].head(10).to_string())

## 2. Análise Temporal por Sorotipo

In [ ]:
# Agregar dados por ano e sorotipo
yearly_serotype = metadata.groupby(['year', 'serotype']).size().unstack(fill_value=0)

# Visualização
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

# Gráfico de linhas
yearly_serotype.plot(kind='line', ax=ax1, marker='o', linewidth=2, markersize=6)
ax1.set_title('Evolução dos Sorotipos de Dengue no Brasil (1986-2012)', fontsize=16, fontweight='bold')
ax1.set_xlabel('Ano', fontsize=12)
ax1.set_ylabel('Número de Sequências', fontsize=12)
ax1.legend(title='Sorotipo', loc='upper left')
ax1.grid(True, alpha=0.3)

# Gráfico de área empilhada
yearly_serotype.plot(kind='area', ax=ax2, alpha=0.7, stacked=True)
ax2.set_title('Distribuição Acumulada de Sorotipos (1986-2012)', fontsize=16, fontweight='bold')
ax2.set_xlabel('Ano', fontsize=12)
ax2.set_ylabel('Número de Sequências', fontsize=12)
ax2.legend(title='Sorotipo', loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Tabela de dados
print("\n📊 Sequências por ano e sorotipo:")
print(yearly_serotype.to_string())

## 3. Proporção de Sorotipos ao Longo do Tempo

In [ ]:
# Calcular proporções
yearly_pct = yearly_serotype.div(yearly_serotype.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 8))

# Gráfico de proporção
yearly_pct.plot(kind='area', ax=ax, alpha=0.7, stacked=True)
ax.set_title('Proporção de Sorotipos ao Longo do Tempo (1986-2012)', fontsize=16, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Proporção (%)', fontsize=12)
ax.legend(title='Sorotipo', loc='upper left', bbox_to_anchor=(1.02, 1))
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

# Adicionar linhas de referência para surtos históricos conhecidos
ax.axvline(x=1998, color='red', linestyle='--', alpha=0.5, label='Surto DENV-1')
ax.axvline(x=2002, color='orange', linestyle='--', alpha=0.5, label='Surto DENV-2')
ax.axvline(x=2007, color='green', linestyle='--', alpha=0.5, label='Surto DENV-3')

plt.tight_layout()
plt.show()

print("\n📊 Proporção por ano (%):")
print(yearly_pct.round(1).to_string())

## 4. Distribuição Geográfica

In [ ]:
# Análise por estado
if 'geoLocAdmin1' in metadata.columns:
    state_analysis = metadata.groupby(['geoLocAdmin1', 'serotype']).size().unstack(fill_value=0)
    state_totals = state_analysis.sum(axis=1).sort_values(ascending=True)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))
    
    # Gráfico de barras empilhadas
    state_analysis.loc[state_totals.index].plot(kind='barh', stacked=True, ax=ax1, alpha=0.8)
    ax1.set_title('Sequências por Estado e Sorotipo', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Número de Sequências', fontsize=12)
    ax1.set_ylabel('Estado/Região', fontsize=12)
    ax1.legend(title='Sorotipo', loc='lower right')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Gráfico de pizza - distribuição geral
    serotype_totals = metadata['serotype'].value_counts()
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    wedges, texts, autotexts = ax2.pie(
        serotype_totals.values, 
        labels=serotype_totals.index,
        autopct='%1.1f%%',
        colors=colors,
        startangle=90
    )
    ax2.set_title('Distribuição Geral por Sorotipo (1986-2012)', fontsize=14, fontweight='bold')
    
    # Tornar texto em negrito
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Total por estado:")
    print(state_totals.sort_values(ascending=False).to_string())

## 5. Análise por Década

In [ ]:
# Análise por década
decade_analysis = metadata.groupby(['decade', 'serotype']).size().unstack(fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Barras agrupadas
decade_analysis.plot(kind='bar', ax=ax1, alpha=0.8)
ax1.set_title('Sequências por Década e Sorotipo', fontsize=14, fontweight='bold')
ax1.set_xlabel('Década', fontsize=12)
ax1.set_ylabel('Número de Sequências', fontsize=12)
ax1.legend(title='Sorotipo')
ax1.set_xticklabels([f"{int(x)}s" for x in decade_analysis.index], rotation=0)
ax1.grid(True, alpha=0.3, axis='y')

# Proporção por década
decade_pct = decade_analysis.div(decade_analysis.sum(axis=1), axis=0) * 100
decade_pct.plot(kind='bar', stacked=True, ax=ax2, alpha=0.8)
ax2.set_title('Proporção por Década (%)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Década', fontsize=12)
ax2.set_ylabel('Proporção (%)', fontsize=12)
ax2.legend(title='Sorotipo', bbox_to_anchor=(1.02, 1))
ax2.set_xticklabels([f"{int(x)}s" for x in decade_pct.index], rotation=0)
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\n📊 Análise por década:")
print(decade_analysis.to_string())
print("\n📊 Proporção por década (%):")
print(decade_pct.round(1).to_string())

## 6. Resumo Estatístico

In [ ]:
print("="*70)
print("📊 RESUMO ESTATÍSTICO - DENGUE NO BRASIL (1986-2012)")
print("="*70)

print(f"\n📈 Geral:")
print(f"  Total de sequências: {len(metadata):,}")
print(f"  Período: {metadata['year'].min()}-{metadata['year'].max()} ({metadata['year'].max() - metadata['year'].min() + 1} anos)")
print(f"  Média anual: {len(metadata) / (metadata['year'].max() - metadata['year'].min() + 1):.0f} sequências")

print(f"\n🦠 Por Sorotipo:")
for st, count in metadata['serotype'].value_counts().items():
    pct = count / len(metadata) * 100
    print(f"  {st}: {count} sequências ({pct:.1f}%)")

print(f"\n📍 Por Estado (Top 5):")
if 'geoLocAdmin1' in metadata.columns:
    for state, count in metadata['geoLocAdmin1'].value_counts().head(5).items():
        pct = count / len(metadata) * 100
        print(f"  {state}: {count} ({pct:.1f}%)")

print(f"\n📅 Picos de Sequenciamento:")
yearly_totals = metadata.groupby('year').size().sort_values(ascending=False)
print(f"  Ano com mais sequências: {yearly_totals.index[0]} ({yearly_totals.iloc[0]} seqs)")
print(f"  Top 3 anos: {', '.join([str(y) for y in yearly_totals.head(3).index])}")

print("="*70)

## 7. Exportar Dados

In [ ]:
# Exportar dados processados
output_file = "pathoplexus_dengue_brasil_1986_2012.csv"
metadata.to_csv(output_file, index=False)
print(f"✅ Dados exportados para: {output_file}")

from pathlib import Path
file_size = Path(output_file).stat().st_size / 1024
print(f"📁 Tamanho: {file_size:.2f} KB")
print(f"📊 Registros: {len(metadata)}")
print(f"📅 Período: {metadata['year'].min()}-{metadata['year'].max()}")

## Conclusões

Este notebook demonstrou a análise de dados genômicos históricos da dengue no Brasil (1986-2012) a partir do Pathoplexus.

### 🎯 Principais Descobertas

1. **Cobertura Temporal:** 26 anos de dados genômicos, com maior concentração na década de 2000

2. **Distribuição de Sorotipos:**
   - **DENV-3** foi o mais prevalente (50.4%) durante o período
   - **DENV-1** e **DENV-2** tiveram circulação significativa
   - **DENV-4** foi o menos comum (7.6%)

3. **Padrões Temporais:** Diferentes sorotipos predominaram em diferentes períodos, indicando
   ciclos de substituição ao longo das décadas

4. **Cobertura Geográfica:** Distribuição em múltiplos estados brasileiros, com concentração
   em regiões de maior incidência histórica

### 💡 Aplicações

- **Evolução Viral:** Análise filogenética de linhagens ao longo do tempo
- **Substituição de Sorotipos:** Estudo da dinâmica de competição entre sorotipos
- **Vigilância Genômica:** Baseline histórico para comparação com dados atuais
- **Correlação Epidemiológica:** Cruzamento com dados de notificação do SINAN

### 📚 Recursos

- Pathoplexus: https://pathoplexus.org/
- API LAPIS: https://lapis.pathoplexus.org/
- InfoDengue: https://info.dengue.mat.br/
- Acessor: `scripts/accessors/pathoplexus.py`